# YOLO11 Instance Segmentation — BCE + Dice Loss

## Objective

This experiment trains YOLO11 instance segmentation using a hybrid mask loss that combines:

- Binary Cross Entropy (BCE) Loss
- Dice Loss

The purpose is to investigate whether combining pixel-level BCE supervision with mask-overlap Dice supervision can improve segmentation performance.

For a fair comparison, this model uses:

- The same original dataset
- The same train/validation/test split
- The same YOLO11n-seg pretrained weights
- The same training hyperparameters
- 100 epochs
- Image size 640
- Batch size 8

The model will be compared against the previously trained BCE-only and Dice-only models.

In [2]:
from pathlib import Path
import torch
import torch.nn.functional as F

from ultralytics import YOLO
from ultralytics.utils.loss import v8SegmentationLoss
from ultralytics.utils.ops import crop_mask

project_path = Path(r"G:\AIIC")

segmentation_project = project_path / "yolo_segmentation"
data_yaml = segmentation_project / "dataset" / "data.yaml"
runs_path = segmentation_project / "runs"

print("Dataset :", data_yaml)
print("Runs    :", runs_path)
print("Dataset exists:", data_yaml.exists())

Dataset : G:\AIIC\yolo_segmentation\dataset\data.yaml
Runs    : G:\AIIC\yolo_segmentation\runs
Dataset exists: True


## Hybrid BCE + Dice Mask Loss

The default YOLO instance segmentation mask loss uses Binary Cross Entropy.

In this experiment, Dice Loss is added to the BCE mask loss.

The final segmentation mask loss is:

Hybrid Loss = 0.5 × BCE Loss + 0.5 × Dice Loss

Equal weighting is used so that both pixel-level accuracy and overall mask overlap contribute to training.

In [3]:
# Save original function once
if not hasattr(v8SegmentationLoss, "_original_single_mask_loss"):
    v8SegmentationLoss._original_single_mask_loss = (
        v8SegmentationLoss.single_mask_loss
    )


def bce_dice_single_mask_loss(
    gt_mask,
    pred,
    proto,
    xyxy,
    area
):
    # ---------------------------------------------------------
    # Reconstruct predicted instance masks
    # ---------------------------------------------------------
    pred_mask = torch.einsum(
        "in,nhw->ihw",
        pred,
        proto
    )

    gt_mask = gt_mask.float()

    # =========================================================
    # 1. BCE LOSS
    # =========================================================

    bce = F.binary_cross_entropy_with_logits(
        pred_mask,
        gt_mask,
        reduction="none"
    )

    bce_loss = (
        crop_mask(bce, xyxy)
        .mean(dim=(1, 2))
        / area
    ).sum()

    # =========================================================
    # 2. DICE LOSS
    # =========================================================

    pred_prob = pred_mask.float().sigmoid()

    # Autograd-safe crop mask
    crop_region = torch.ones_like(pred_prob)
    crop_region = crop_mask(
        crop_region,
        xyxy
    )

    pred_crop = pred_prob * crop_region
    gt_crop = gt_mask * crop_region

    intersection = (
        pred_crop * gt_crop
    ).sum(dim=(1, 2))

    pred_sum = pred_crop.sum(dim=(1, 2))
    gt_sum = gt_crop.sum(dim=(1, 2))

    eps = 1e-7

    dice_score = (
        2.0 * intersection + eps
    ) / (
        pred_sum + gt_sum + eps
    )

    dice_loss = (
        1.0 - dice_score
    ).sum()

    # =========================================================
    # 3. HYBRID LOSS
    # =========================================================

    hybrid_loss = (
        0.5 * bce_loss
        +
        0.5 * dice_loss
    )

    return hybrid_loss


v8SegmentationLoss.single_mask_loss = staticmethod(
    bce_dice_single_mask_loss
)

print("✅ BCE + Dice hybrid mask loss activated.")
print(
    "Current function:",
    v8SegmentationLoss.single_mask_loss.__name__
)

✅ BCE + Dice hybrid mask loss activated.
Current function: bce_dice_single_mask_loss


## Verify Hybrid Loss

This cell confirms that the custom segmentation loss contains both BCE and Dice components before training begins.

In [4]:
import inspect

source = inspect.getsource(
    v8SegmentationLoss.single_mask_loss
)

print(source)

assert "binary_cross_entropy_with_logits" in source
assert "dice_score" in source

print()
print("✅ BCE component found")
print("✅ Dice component found")
print("✅ Hybrid loss verification passed")

def bce_dice_single_mask_loss(
    gt_mask,
    pred,
    proto,
    xyxy,
    area
):
    # ---------------------------------------------------------
    # Reconstruct predicted instance masks
    # ---------------------------------------------------------
    pred_mask = torch.einsum(
        "in,nhw->ihw",
        pred,
        proto
    )

    gt_mask = gt_mask.float()

    # =========================================================
    # 1. BCE LOSS
    # =========================================================

    bce = F.binary_cross_entropy_with_logits(
        pred_mask,
        gt_mask,
        reduction="none"
    )

    bce_loss = (
        crop_mask(bce, xyxy)
        .mean(dim=(1, 2))
        / area
    ).sum()

    # =========================================================
    # 2. DICE LOSS
    # =========================================================

    pred_prob = pred_mask.float().sigmoid()

    # Autograd-safe crop mask
    crop_region = torch.ones_like(pred_

## Train YOLO11n-Seg with BCE + Dice Loss

A fresh pretrained YOLO11n segmentation model is used to ensure a fair comparison with the BCE-only and Dice-only experiments.

In [5]:
model_bce_dice = YOLO("yolo11n-seg.pt")

bce_dice_results = model_bce_dice.train(
    data=str(data_yaml),

    epochs=100,
    imgsz=640,
    batch=8,

    device=0,
    workers=4,

    patience=20,

    pretrained=True,
    optimizer="auto",

    amp=True,
    cache=False,

    project=str(runs_path),
    name="yolo11n_seg_aiic_bce_dice_original",

    exist_ok=False,

    plots=True,
    verbose=True
)

New https://pypi.org/project/ultralytics/8.4.144 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.137  Python-3.14.3 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060, 8123MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=G:\AIIC\yolo_segmentation\dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=3